# Pre-training vs training da zero vs fine-tuning

## Obiettivo didattico

In questo esercizio confrontiamo tre modi di ottenere un modello di embedding per un **dominio specifico** (l'esplorazione spaziale), con pochissimi dati a disposizione:

1. **Training da zero ("cold start")** — alleniamo Word2Vec solo sul nostro piccolo corpus di dominio, senza nessuna conoscenza pregressa.
2. **Modello pre-addestrato puro** — usiamo un modello già addestrato su un corpus enorme e generico (Google News), senza alcun adattamento al nostro dominio.
3. **Fine-tuning ("warm start")** — partiamo dai vettori pre-addestrati e li adattiamo continuando il training sul nostro piccolo corpus di dominio.

L'idea è la stessa che sta dietro a modelli come BERT o GPT: **pre-training su tanti dati generici + fine-tuning su pochi dati specifici** batte quasi sempre il training da zero su pochi dati specifici.

In [2]:
import gensim.downloader as api
from gensim.models import Word2Vec
import numpy as np

# Corpus di dominio: poche frasi sull'esplorazione spaziale.
# Notare quanto è piccolo: 8 frasi, vocabolario minuscolo.
domain_corpus = [
    ['telescope', 'observes', 'distant', 'galaxy', 'through', 'space'],
    ['astronaut', 'orbits', 'earth', 'aboard', 'space', 'station'],
    ['rocket', 'launches', 'satellite', 'into', 'orbit'],
    ['galaxy', 'contains', 'billions', 'of', 'stars', 'and', 'planets'],
    ['mission', 'control', 'tracks', 'the', 'rocket', 'trajectory'],
    ['astronaut', 'walks', 'on', 'the', 'moon', 'surface'],
    ['telescope', 'captures', 'images', 'of', 'nebula'],
    ['space', 'station', 'orbits', 'earth', 'every', 'ninety', 'minutes'],
]

vocab_size = len({w for sent in domain_corpus for w in sent})
print(f"Numero di frasi nel corpus di dominio: {len(domain_corpus)}")
print(f"Dimensione del vocabolario di dominio: {vocab_size} parole")

Numero di frasi nel corpus di dominio: 8
Dimensione del vocabolario di dominio: 37 parole


## Parte 1 — Training da zero (nessun pre-training)

Alleniamo un modello Word2Vec **solo** su queste 8 frasi. Nessuna conoscenza pregressa: i vettori iniziali sono casuali e tutto quello che il modello "sa" viene esclusivamente da questo minuscolo corpus.

In [3]:
model_scratch = Word2Vec(
    domain_corpus,
    vector_size=100,
    window=3,#il modello considera 3 parole prima e 3 parole dopo la parola presa in considerazione
    min_count=1, #considera tutte le parole ripetute almeno 1 volta
    sg=1, #In Word2Vec (come nell'implementazione di Gensim Word2Vec), il parametro sg=1 serve ad attivare l'architettura Skip-gram al posto della predefinita CBOW (sg=0).
    epochs=50,
    seed=42,
)

#Come funziona sg=1 (Skip-gram)Obiettivo: Dato un termine centrale (target), il modello prova a predire le parole del 
# contesto circostante (le parole che compaiono prima e dopo nel testo).Differenza con sg=0 (CBOW):
#  CBOW fa il contrario, cioè predice la parola centrale partendo dalle parole del contesto.

print("Parole più simili a 'rocket' (modello da zero):")
for w, score in model_scratch.wv.most_similar('rocket', topn=5):
    print(f"  {w:12s} {score:.3f}") 

print("\nParole più simili a 'astronaut' (modello da zero):")
for w, score in model_scratch.wv.most_similar('astronaut', topn=5):
    print(f"  {w:12s} {score:.3f}")

Parole più simili a 'rocket' (modello da zero):
  minutes      0.169
  observes     0.168
  stars        0.162
  ninety       0.113
  of           0.111

Parole più simili a 'astronaut' (modello da zero):
  minutes      0.186
  orbits       0.163
  and          0.133
  rocket       0.104
  walks        0.097


**Domanda per la classe:** i vicini di 'rocket' e 'astronaut' hanno senso? Probabilmente no, o solo in parte — con 8 frasi il modello non ha visto abbastanza contesti diversi per imparare relazioni semantiche solide. Questo è il problema classico del training da zero quando i dati sono pochi: **il modello sovradatta al piccolo corpus e non generalizza**.

## Parte 2 — Il modello pre-addestrato

Ora carichiamo un modello Word2Vec addestrato da Google su ~100 miliardi di parole di notizie. Non lo abbiamo addestrato noi: lo stiamo semplicemente **riutilizzando**. Questo è il concetto di **pre-training**: qualcuno (con molte più risorse di calcolo e dati di quelle che abbiamo noi) ha già fatto il lavoro pesante una volta, su un corpus enorme e generico.

In [4]:
pretrained = api.load("word2vec-google-news-300")

print("Parole più simili a 'rocket' (modello pre-addestrato, dominio generico):")
for w, score in pretrained.most_similar('rocket', topn=5):
    print(f"  {w:15s} {score:.3f}")

Parole più simili a 'rocket' (modello pre-addestrato, dominio generico):
  rockets         0.820
  missile         0.677
  Qassam_rocket   0.618
  Katyusha_rocket 0.614
  Kassam_rocket   0.600


**Domanda per la classe:** questi vicini hanno più senso di quelli del modello da zero? Probabilmente sì — ma notare che questo modello **non conosce il nostro corpus specifico** (non ha mai visto le nostre 8 frasi), quindi non è specializzato sul nostro dominio. È "intelligente in generale" ma non su misura per noi.

## Parte 3 — Fine-tuning: il meglio dei due mondi

Ora combiniamo le due cose:

1. Creiamo un nuovo modello Word2Vec e costruiamo il vocabolario dal nostro corpus di dominio (come nella Parte 1).
2. **Invece di partire da vettori casuali**, inizializziamo ogni parola in comune con il suo vettore pre-addestrato (preso dal modello di Google News).
3. Continuiamo il training (poche epoche) **solo** sul nostro corpus di dominio, per adattare i vettori al contesto specifico.

Questo è, in miniatura, lo stesso principio del fine-tuning di modelli come BERT o GPT: si parte da pesi già addestrati su tantissimi dati generici, e si adattano con pochi dati specifici del task.

In [ ]:
# 1. Creiamo il modello e costruiamo il vocabolario dal corpus di dominio
model_finetuned = Word2Vec(
    vector_size=300,   # deve coincidere con la dimensione dei vettori pre-addestrati
    window=3,
    min_count=1,
    sg=1,
    seed=42,
)
model_finetuned.build_vocab(domain_corpus)

# 2. "Warm start": copiamo i vettori pre-addestrati per le parole in comune
n_initialized = 0
for word in model_finetuned.wv.index_to_key:
    if word in pretrained:
        model_finetuned.wv[word] = pretrained[word]
        n_initialized += 1
#quindi se la parola è presente nel modello preaddestrato, sarà una parola (gia) inizializzata

print(f"Parole inizializzate con vettori pre-addestrati: {n_initialized} / {len(model_finetuned.wv.index_to_key)}")

# 3. Fine-tuning: continuiamo il training sul piccolo corpus di dominio
model_finetuned.train(domain_corpus, total_examples=model_finetuned.corpus_count, epochs=50)

print("\nParole più simili a 'rocket' (modello fine-tuned):")
for w, score in model_finetuned.wv.most_similar('rocket', topn=5):
    print(f"  {w:15s} {score:.3f}")

print("\nParole più simili a 'astronaut' (modello fine-tuned):")
for w, score in model_finetuned.wv.most_similar('astronaut', topn=5):
    print(f"  {w:15s} {score:.3f}")

Parole inizializzate con vettori pre-addestrati: 35 / 37

Parole più simili a 'rocket' (modello fine-tuned):
  and             0.473
  orbit           0.470
  of              0.466
  astronaut       0.390
  moon            0.374

Parole più simili a 'astronaut' (modello fine-tuned):
  orbit           0.529
  and             0.463
  of              0.454
  moon            0.431
  rocket          0.390


## Parte 4 — Confronto finale

Mettiamo a confronto i tre approcci sulla stessa domanda: *quali parole sono più simili a 'space'?*

In [6]:
target_word = 'space'

print("="*60)
print(f"Confronto per la parola: '{target_word}'")
print("="*60)

print("\n[1] Training da zero (solo 8 frasi, nessun pre-training):")
for w, score in model_scratch.wv.most_similar(target_word, topn=5):
    print(f"    {w:15s} {score:.3f}")

print("\n[2] Modello pre-addestrato puro (Google News, senza fine-tuning):")
for w, score in pretrained.most_similar(target_word, topn=5):
    print(f"    {w:15s} {score:.3f}")

print("\n[3] Pre-training + fine-tuning sul nostro dominio:")
for w, score in model_finetuned.wv.most_similar(target_word, topn=5):
    print(f"    {w:15s} {score:.3f}")

Confronto per la parola: 'space'

[1] Training da zero (solo 8 frasi, nessun pre-training):
    moon            0.220
    galaxy          0.196
    observes        0.158
    planets         0.150
    images          0.143

[2] Modello pre-addestrato puro (Google News, senza fine-tuning):
    spaces          0.657
    music_concept_ShockHound 0.585
    Shuttle_docks   0.557
    Space           0.548
    Soviet_Union_Yuri_Gagarin 0.542

[3] Pre-training + fine-tuning sul nostro dominio:
    of              0.470
    and             0.434
    orbit           0.423
    astronaut       0.384
    moon            0.357


## Domande di discussione per la classe

1. **Perché il modello da zero (approccio 1) fatica a produrre vicini sensati?** Cosa cambierebbe con un corpus 1000 volte più grande?
2. **Il modello pre-addestrato puro (approccio 2) conosce il nostro dominio specifico?** Cosa succederebbe se cercassimo la parola "nebula" o "cosmonauta" (termini più di nicchia)?
3. **Cosa ha "guadagnato" il modello fine-tuned (approccio 3) rispetto agli altri due?** In che senso ha sia conoscenza generale sia adattamento al dominio?
4. **Catastrophic forgetting**: continuando il training molto a lungo *solo* sul piccolo corpus di dominio, cosa rischia di succedere alla conoscenza generale ereditata dal pre-training? (Suggerimento: con poche epoche il modello si adatta senza "disimparare" troppo; con troppe epoche rischia di sovrascrivere quella conoscenza.)
5. **Collegamento con i modelli linguistici moderni**: BERT e GPT vengono pre-addestrati su miliardi di parole di testo generico, poi vengono fine-tuned (o usati via prompting) su task specifici con pochissimi esempi. Quali analogie vedete con questo esperimento in miniatura?

## Esercizio per gli studenti

- Modifica `domain_corpus` con un dominio a vostra scelta (es. cucina, sport, medicina).
- Riesegui i tre modelli e confronta i vicini di 3-4 parole chiave del vostro dominio.
- Provate a variare `epochs` nel fine-tuning (es. 5 invece di 50): cosa succede ai risultati? Il modello fine-tuned si comporta più come il modello pre-addestrato puro o più come quello da zero?

In [7]:
# Corpus di dominio: poche frasi sul tema della cucina e gastronomia.
# Notare quanto è piccolo: 8 frasi, vocabolario minuscolo.
domain_corpus2 = [
    ['chef', 'prepares', 'fresh', 'pasta', 'in', 'kitchen'],
    ['cook', 'bakes', 'delicious', 'pizza', 'inside', 'oven'],
    ['recipe', 'requires', 'olive', 'oil', 'and', 'fresh', 'basil'],
    ['pizza', 'contains', 'tomato', 'sauce', 'and', 'melted', 'mozzarella'],
    ['kitchen', 'staff', 'prepares', 'the', 'sauce', 'daily'],
    ['chef', 'cooks', 'risotto', 'on', 'the', 'stovetop'],
    ['oven', 'bakes', 'crusty', 'bread', 'for', 'dinner'],
    ['italian', 'restaurant', 'serves', 'fresh', 'pasta', 'every', 'evening'],
]

vocab_size2 = len({w for sent in domain_corpus2 for w in sent})
print(f"Numero di frasi nel corpus di dominio: {len(domain_corpus2)}")
print(f"Dimensione del vocabolario di dominio: {vocab_size2} parole")

# 1. Creiamo il modelli
model_scratch2 = Word2Vec(
    domain_corpus2,
    vector_size=100,
    window=3,#il modello considera 3 parole prima e 3 parole dopo la parola presa in considerazione
    min_count=1, #considera tutte le parole ripetute almeno 1 volta
    sg=1, 
    epochs=50,
    seed=42,
)


print("Parole più simili a 'cooks' (modello da zero):")
for w, score in model_scratch2.wv.most_similar('cooks', topn=5):
    print(f"  {w:12s} {score:.3f}") 

print("\nParole più simili a 'chef' (modello da zero):")
for w, score in model_scratch2.wv.most_similar('chef', topn=5):
    print(f"  {w:12s} {score:.3f}")



Numero di frasi nel corpus di dominio: 8
Dimensione del vocabolario di dominio: 39 parole
Parole più simili a 'cooks' (modello da zero):
  and          0.091
  dinner       0.085
  contains     0.078
  serves       0.067
  sauce        0.061

Parole più simili a 'chef' (modello da zero):
  bakes        0.316
  requires     0.196
  kitchen      0.188
  the          0.175
  and          0.174


In [8]:
pretrained = api.load("word2vec-google-news-300")

print("Parole più simili a 'cooks' (modello pre-addestrato, dominio generico):")
for w, score in pretrained.most_similar('cooks', topn=5):
    print(f"  {w:15s} {score:.3f}")

print("\nParole più simili a 'chef' (modello pre-addestrato, dominio generico):")
for w, score in pretrained.most_similar('chef', topn=5):
    print(f"  {w:15s} {score:.3f}")

Parole più simili a 'cooks' (modello pre-addestrato, dominio generico):
  cook            0.758
  chefs           0.699
  cooking         0.658
  bakers          0.622
  dishes          0.589

Parole più simili a 'chef' (modello pre-addestrato, dominio generico):
  Chef            0.823
  pastry_chef     0.795
  sous_chef       0.793
  chefs           0.780
  Executive_Chef  0.734


In [9]:
# 1. Creiamo il modello e costruiamo il vocabolario dal corpus di dominio
model_finetuned2 = Word2Vec(
    vector_size=300,   # deve coincidere con la dimensione dei vettori pre-addestrati
    window=3,
    min_count=1,
    sg=1,
    seed=42,
)
model_finetuned2.build_vocab(domain_corpus2)

# 2. "Warm start": copiamo i vettori pre-addestrati per le parole in comune
n_initialized = 0
for word in model_finetuned2.wv.index_to_key:
    if word in pretrained:
        model_finetuned2.wv[word] = pretrained[word]
        n_initialized += 1
#quindi se la parola è presente nel modello preaddestrato, sarà una parola (gia) inizializzata

print(f"Parole inizializzate con vettori pre-addestrati: {n_initialized} / {len(model_finetuned2.wv.index_to_key)}")

# 3. Fine-tuning: continuiamo il training sul piccolo corpus di dominio
model_finetuned.train(domain_corpus2, total_examples=model_finetuned2.corpus_count, epochs=50)

print("\nParole più simili a 'cooks' (modello fine-tuned):")
for w, score in model_finetuned2.wv.most_similar('cooks', topn=5):
    print(f"  {w:15s} {score:.3f}")

print("\nParole più simili a 'chef' (modello fine-tuned):")
for w, score in model_finetuned2.wv.most_similar('chef', topn=5):
    print(f"  {w:15s} {score:.3f}")



Parole inizializzate con vettori pre-addestrati: 38 / 39

Parole più simili a 'cooks' (modello fine-tuned):
  cook            0.758
  chef            0.577
  kitchen         0.558
  bakes           0.514
  recipe          0.486

Parole più simili a 'chef' (modello fine-tuned):
  cooks           0.577
  cook            0.563
  restaurant      0.557
  risotto         0.518
  kitchen         0.507


In [10]:
# Corpus di dominio: poche frasi sul tema della cucina e gastronomia.
# Notare quanto è piccolo: 8 frasi, vocabolario minuscolo.
domain_corpus2 = [
    ['chef', 'prepares', 'fresh', 'pasta', 'in', 'kitchen'],
    ['cook', 'bakes', 'delicious', 'pizza', 'inside', 'oven'],
    ['recipe', 'requires', 'olive', 'oil', 'and', 'fresh', 'basil'],
    ['pizza', 'contains', 'tomato', 'sauce', 'and', 'melted', 'mozzarella'],
    ['kitchen', 'staff', 'prepares', 'the', 'sauce', 'daily'],
    ['chef', 'cooks', 'risotto', 'on', 'the', 'stovetop'],
    ['oven', 'bakes', 'crusty', 'bread', 'for', 'dinner'],
    ['italian', 'restaurant', 'serves', 'fresh', 'pasta', 'every', 'evening'],
]

vocab_size2 = len({w for sent in domain_corpus2 for w in sent})
print(f"Numero di frasi nel corpus di dominio: {len(domain_corpus2)}")
print(f"Dimensione del vocabolario di dominio: {vocab_size2} parole")

# 1. Creiamo il modelli
model_scratch2 = Word2Vec(
    domain_corpus2,
    vector_size=100,
    window=3,#il modello considera 3 parole prima e 3 parole dopo la parola presa in considerazione
    min_count=1, #considera tutte le parole ripetute almeno 1 volta
    sg=1, 
    epochs=5,
    seed=42,
)


print("Parole più simili a 'cooks' (modello da zero):")
for w, score in model_scratch2.wv.most_similar('cooks', topn=5):
    print(f"  {w:12s} {score:.3f}") 

print("\nParole più simili a 'chef' (modello da zero):")
for w, score in model_scratch2.wv.most_similar('chef', topn=5):
    print(f"  {w:12s} {score:.3f}")


print("="*52)

pretrained = api.load("word2vec-google-news-300")

print("Parole più simili a 'cooks' (modello pre-addestrato, dominio generico):")
for w, score in pretrained.most_similar('cooks', topn=5):
    print(f"  {w:15s} {score:.3f}")

print("\nParole più simili a 'chef' (modello pre-addestrato, dominio generico):")
for w, score in pretrained.most_similar('chef', topn=5):
    print(f"  {w:15s} {score:.3f}")

print("="*52)
# 1. Creiamo il modello e costruiamo il vocabolario dal corpus di dominio
model_finetuned2 = Word2Vec(
    vector_size=300,   # deve coincidere con la dimensione dei vettori pre-addestrati
    window=3,
    min_count=1,
    sg=1,
    seed=42,
)
model_finetuned2.build_vocab(domain_corpus2)

# 2. "Warm start": copiamo i vettori pre-addestrati per le parole in comune
n_initialized = 0
for word in model_finetuned2.wv.index_to_key:
    if word in pretrained:
        model_finetuned2.wv[word] = pretrained[word]
        n_initialized += 1
#quindi se la parola è presente nel modello preaddestrato, sarà una parola (gia) inizializzata

print(f"Parole inizializzate con vettori pre-addestrati: {n_initialized} / {len(model_finetuned2.wv.index_to_key)}")

# 3. Fine-tuning: continuiamo il training sul piccolo corpus di dominio
model_finetuned.train(domain_corpus2, total_examples=model_finetuned2.corpus_count, epochs=50)

print("\nParole più simili a 'cooks' (modello fine-tuned):")
for w, score in model_finetuned2.wv.most_similar('cooks', topn=5):
    print(f"  {w:15s} {score:.3f}")

print("\nParole più simili a 'chef' (modello fine-tuned):")
for w, score in model_finetuned2.wv.most_similar('chef', topn=5):
    print(f"  {w:15s} {score:.3f}")

Numero di frasi nel corpus di dominio: 8
Dimensione del vocabolario di dominio: 39 parole
Parole più simili a 'cooks' (modello da zero):
  and          0.095
  dinner       0.085
  contains     0.083
  serves       0.072
  sauce        0.067

Parole più simili a 'chef' (modello da zero):
  bakes        0.313
  requires     0.192
  kitchen      0.186
  the          0.170
  and          0.169
Parole più simili a 'cooks' (modello pre-addestrato, dominio generico):
  cook            0.758
  chefs           0.699
  cooking         0.658
  bakers          0.622
  dishes          0.589

Parole più simili a 'chef' (modello pre-addestrato, dominio generico):
  Chef            0.823
  pastry_chef     0.795
  sous_chef       0.793
  chefs           0.780
  Executive_Chef  0.734
Parole inizializzate con vettori pre-addestrati: 38 / 39

Parole più simili a 'cooks' (modello fine-tuned):
  cook            0.758
  chef            0.577
  kitchen         0.558
  bakes           0.514
  recipe          

 A prima vista sembra che il fine-tuning produca risultati peggiori rispetto al modello addestrato da zero o a quello generico, restituendo parole irrilevanti come "and" o "of".

In realtà, il modello sta funzionando esattamente per come l'hai configurato, ma ha subito un fenomeno chiamato Catastrophic Forgetting (dimenticanza catastrofica) unito a un forte Overfitting sul rumore del micro-corpus.

Ecco i 4 motivi principali spiegati in modo semplice per la tua presentazione in classe:

1. La trappola delle Stopwords ("and", "of")
Abbiamo incluso parole grammaticali (stopwords) come "and", "of", "into", "the".

Queste parole compaiono in quasi tutte le frasi vicino a termini importanti come rocket, astronaut e space.

Il risultato: Il modello impara che "and" e "of" appaiono sempre nel contesto di rocket e astronaut, assegnando loro una somiglianza coseno altissima.

Come si risolve di solito: Nei corpus reali, queste parole vengono rimosse (preprocessing) oppure si usa il parametro sample di Gensim per ridurne il peso.

2. Overfitting estremo su un corpus "microscopico"
Per addestrare (o fare fine-tuning di) una rete neurale come Word2Vec servono solitamente milioni di frasi.

Hai eseguito 50 epoche (epochs=50) su sole 8 frasi.

È come costringere uno studente a rileggere gli stessi 8 elementi per 50 volte di seguito: invece di capire il concetto generale, memorizza la struttura esatta delle frasi.

3. Il problema dell'inizializzazione parziale (Cold/Warm Weights)

Il modello fine-tuned ha completamente sovrascritto le buone rappresentazioni originali di Google News, "distruggendo" lo spazio vettoriale precedente per adattarsi alle sole 8 frasi.

Word2Vec non è fatto solo dai vettori di output che leggi con .wv (i target vectors), ma possiede anche una seconda matrice interna di pesi (i context vectors nella rete neurale).

Iniettando manualmente i vettori solo nella matrice di output .wv, la matrice interna del contesto è rimasta inizializzata casualmente.

Quando hai chiamato .train(), l'ottimizzatore ha dovuto adattare i vettori pre-addestrati a una matrice di contesto totalmente casuale, scombussolando le relazioni geometriche apprese da Google News.




In [13]:
import gensim.downloader as api
from gensim.models import Word2Vec

# 1. Carichiamo il modello pre-addestrato
print("Caricamento modello Google News in corso...")
pretrained = api.load("word2vec-google-news-300")

# 2. Corpus pulito (SENZA stopwords)
domain_corpus_clean = [
    ['telescope', 'observes', 'distant', 'galaxy', 'space'],
    ['astronaut', 'orbits', 'earth', 'aboard', 'space', 'station'],
    ['rocket', 'launches', 'satellite', 'orbit'],
    ['galaxy', 'contains', 'billions', 'stars', 'planets'],
    ['mission', 'control', 'tracks', 'rocket', 'trajectory'],
    ['astronaut', 'walks', 'moon', 'surface'],
    ['telescope', 'captures', 'images', 'nebula'],
    ['space', 'station', 'orbits', 'earth', 'ninety', 'minutes'],
]

# 3. Creiamo il modello Word2Vec di destinazione
model_finetuned = Word2Vec(
    vector_size=300,
    window=3,
    min_count=1,
    sg=1,
    seed=42,
    alpha=0.001,       # Learning rate MOLTO basso: vogliamo solo un piccolo ritocco
    min_alpha=0.0001
)

# Costruiamo il vocabolario per il nostro dominio
model_finetuned.build_vocab(domain_corpus_clean)

# 4. TRASFERIMENTO PESI (Compatibile al 100% con Gensim 4.x)
# Copiamo i vettori da Google News per le parole che esistono in entrambi
n_migrated = 0
for word in model_finetuned.wv.index_to_key:
    if word in pretrained:
        model_finetuned.wv[word] = pretrained[word]
        n_migrated += 1

print(f"Inizializzate {n_migrated} parole su {len(model_finetuned.wv.index_to_key)} con i pesi di Google News.")

# 5. FINE-TUNING LEGGERISSIMO
# Solo 2 epoche: evitano che la matrice dei contesti (random) distrugga i vettori coprati
model_finetuned.train(
    domain_corpus_clean, 
    total_examples=model_finetuned.corpus_count, 
    epochs=2
)

# --- VERIFICA DEI RISULTATI ---
target_word = 'space'
print("\n" + "="*50)
print(f"Parole più simili a '{target_word}' dopo il Fine-Tuning corretto:")
print("="*50)

for w, score in model_finetuned.wv.most_similar(target_word, topn=5):
    print(f"   {w:15s} {score:.3f}")

Caricamento modello Google News in corso...
Inizializzate 30 parole su 30 con i pesi di Google News.

Parole più simili a 'space' dopo il Fine-Tuning corretto:
   orbit           0.416
   astronaut       0.379
   moon            0.348
   satellite       0.299
   rocket          0.297
